In [52]:
import pandas as pd
import numpy as np

In [5]:
# Einlesen der csv Datei
df = pd.read_csv("E-commerce Customer Behavior - Sheet1.csv")

In [6]:
df.head()

,Customer ID,Gender,Age,City,Membership Type,Total Spend,Items Purchased,Average Rating,Discount Applied,Days Since Last Purchase,Satisfaction Level
0,101,Female,29,New York,Gold,1120.20,14,4.6,True,25,Satisfied
1,102,Male,34,Los Angeles,Silver,780.50,11,4.1,False,18,Neutral
2,103,Female,43,Chicago,Bronze,510.75,9,3.4,True,42,Unsatisfied
3,104,Male,30,San Francisco,Gold,1480.30,19,4.7,False,12,Satisfied
4,105,Male,27,Miami,Silver,720.40,13,4.0,True,55,Unsatisfied


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 350 entries, 0 to 349
Data columns (total 11 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Customer ID               350 non-null    int64  
 1   Gender                    350 non-null    str    
 2   Age                       350 non-null    int64  
 3   City                      350 non-null    str    
 4   Membership Type           350 non-null    str    
 5   Total Spend               350 non-null    float64
 6   Items Purchased           350 non-null    int64  
 7   Average Rating            350 non-null    float64
 8   Discount Applied          350 non-null    bool   
 9   Days Since Last Purchase  350 non-null    int64  
 10  Satisfaction Level        348 non-null    str    
dtypes: bool(1), float64(2), int64(4), str(4)
memory usage: 27.8 KB


In [8]:
df.describe()

,Customer ID,Age,Total Spend,Items Purchased,Average Rating,Days Since Last Purchase
count,350.000000,350.000000,350.000000,350.000000,350.000000,350.000000
mean,275.500000,33.597143,845.381714,12.600000,4.019143,26.588571
std,101.180532,4.870882,362.058695,4.155984,0.580539,13.440813
min,101.000000,26.000000,410.800000,7.000000,3.000000,9.000000
25%,188.250000,30.000000,502.000000,9.000000,3.500000,15.000000
50%,275.500000,32.500000,775.200000,12.000000,4.100000,23.000000
75%,362.750000,37.000000,1160.600000,15.000000,4.500000,38.000000
max,450.000000,43.000000,1520.100000,21.000000,4.900000,63.000000


In [10]:
df.isnull().sum()

Customer ID                 0
Gender                      0
Age                         0
City                        0
Membership Type             0
Total Spend                 0
Items Purchased             0
Average Rating              0
Discount Applied            0
Days Since Last Purchase    0
Satisfaction Level          2
dtype: int64

In [11]:
# Spaltennamen für die weitere Analyse bereinigen
df.columns =df.columns.str.lower()
df.columns =df.columns.str.replace(" ", "_")

In [12]:
df.columns

Index(['customer_id', 'gender', 'age', 'city', 'membership_type',
       'total_spend', 'items_purchased', 'average_rating', 'discount_applied',
       'days_since_last_purchase', 'satisfaction_level'],
      dtype='str')

In [13]:
df[df["satisfaction_level"].isnull()]

,customer_id,gender,age,city,membership_type,total_spend,items_purchased,average_rating,discount_applied,days_since_last_purchase,satisfaction_level
71,172,Female,37,Houston,Bronze,420.8,7,3.1,False,21,NaN
143,244,Female,37,Houston,Bronze,430.8,7,3.4,False,23,NaN


In [14]:
df["satisfaction_level"].describe()

count           348
unique            3
top       Satisfied
freq            125
Name: satisfaction_level, dtype: object

In [19]:
# Fehlende Werte in satisfaction_level durch die häufigste Kategorie("Satisfied") ersetzen
df["satisfaction_level"] =df["satisfaction_level"].fillna("Satisfied")

In [20]:
df.isnull().any()

customer_id                 False
gender                      False
age                         False
city                        False
membership_type             False
total_spend                 False
items_purchased             False
average_rating              False
discount_applied            False
days_since_last_purchase    False
satisfaction_level          False
dtype: bool

In [28]:
# Kunden nach Alter in Altersgruppen einteilen 
group =["26-30","31-35","36-40","41-45"]
df["age_group"] = pd.cut(df["age"], bins=[25,30,35,40,45], labels=group)

In [30]:
df["age_group"].unique()

['26-30', '31-35', '41-45', '36-40']
Categories (4, str): ['26-30' < '31-35' < '36-40' < '41-45']

In [46]:
# Kunden anhand ihrere Gesamtausgaben in drei gleich große Gruppen
# (Low, Medium, High Spender) einteilen
df["spending_level"], bins = pd.qcut(df["total_spend"],q=3, labels=["Low", "Medium", "High"], retbins=True)

In [43]:
df["spending_level"].unique()

['High', 'Medium', 'Low']
Categories (3, str): ['Low' < 'Medium' < 'High']

In [54]:
print(bins.round(2))

[ 410.8   660.3  1023.77 1520.1 ]


In [57]:
from sqlalchemy import create_engine

username = "root"
password = "password"
host = "localhost"
port = "3306"
database = "ecommerce_analysis"

engine = create_engine(
    f"mysql+pymysql://{username}:{password}@{host}:{port}/{database}"
)

df.to_sql(
    "customer",
    engine,
    if_exists="replace",
    index=False
)

print("Daten erfolgreich geladen")

Daten erfolgreich geladen
